# Week 89, Day 5: Code review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w089d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('planner.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\nimport math\n\n\ndef straight_km(positions, a, b):\n    """Straight-line distance in km between two junctions."""\n    return math.dist(positions[a], positions[b])\n\n\ndef astar(paths, positions, start, end):\n    g, prev, done = {start: 0}, {}, set()\n    heap = [(straight_km(positions, start, end), start)]\n    while heap:\n        _, node = heapq.heappop(heap)\n        if node in done:\n            continue\n        done.add(node)\n        if node == end:\n            stops = [end]\n            while stops[-1] != start:\n                stops.append(prev[stops[-1]])\n            return stops[::-1], g[end]\n        for nxt, km in sorted(paths[node].items()):\n            if g[node] + km < g.get(nxt, float("inf")):\n                g[nxt] = g[node] + km\n                prev[nxt] = node\n                heapq.heappush(heap, (g[nxt] + straight_km(positions, nxt, end), nxt))\n    return None\n\n\ndef describe(stops: list, km: float, minutes: int) -> str:\n    """One line for the command-line tool."""\n    hours, mins = divmod(minutes, 60)\n    return f"{\' -> \'.join(stops)}: {km:.1f} km, {hours} h {mins:02d} min"\n\n\ndef pick(routes, prefer):\n    if prefer == "short" or prefer == "shortest":\n        return min(routes, key=lambda r: r["km"])\n    elif prefer == "flat" and routes:\n        return min(routes, key=lambda r: r["climb"])\n    elif prefer == "quiet":\n        quiet = [r for r in routes if r["road_km"] == 0]\n        if quiet:\n            return quiet[0]\n        else:\n            return min(routes, key=lambda r: r["road_km"])\n    return routes[0] if routes else None\n\n\ndef _cache_key(start, end):\n    return f"{start}|{end}"\n')
with open('test_planner.py', 'w', encoding='utf-8') as f:
    f.write('import random\nimport time\nimport pytest\nfrom planner import astar, describe, pick\n\n\ndef test_describe():\n    assert describe(["A", "B"], 1.0, 75) == "A -> B: 1.0 km, 1 h 15 min"\n\n\ndef test_astar_runs():\n    astar({"A": {"B": 1}, "B": {"A": 1}}, {"A": (0, 0), "B": (1, 0)}, "A", "B")\n\n\ndef test_pick_waits_for_cache():\n    time.sleep(0.01)\n    assert pick([{"km": 2, "climb": 0, "road_km": 0}], "short")["km"] == 2\n\n\ndef test_pick_errors():\n    with pytest.raises(Exception):\n        pick([], "flat")["km"]\n\n\ndef test_random_routes():\n    rng = random.Random()\n    routes = [{"km": rng.random(), "climb": 0, "road_km": 0} for _ in range(5)]\n    assert pick(routes, "short")["km"] == min(r["km"] for r in routes)\n')

**Predict before you run:** what will this print?



In [ ]:
import ast

source = open("planner.py").read()
tree = ast.parse(source)
for node in tree.body:
    if isinstance(node, ast.FunctionDef):
        branches = sum(isinstance(n, (ast.If, ast.For, ast.While)) for n in ast.walk(node))
        print(f"{node.name:12} lines {node.end_lineno - node.lineno + 1:2}  branches {branches}  docstring {ast.get_docstring(node) is not None}")

> **Key idea.** Review algorithm changes for correctness on awkward inputs, cost (loops inside loops, growth tests), written and checked assumptions, and containment. Measure cyclomatic complexity (1 plus each decision) and length from the syntax tree; flag functions above about 10 branches or 30 to 40 lines, and simplify with tables or smaller functions. Public functions need docstrings and type hints. Review tests for smells: no assertion, sleeps, `pytest.raises(Exception)`, unseeded randomness, testing mocks. Automate the measurements in CI, and spend the review on judgement.

### Exercise: Cyclomatic complexity

Write `complexity(source)`: a dict from the name of each top-level function in the module source to its cyclomatic complexity: 1, plus 1 for each `if` (including `elif`, which is an `If` node), `for`, `while`, `except` handler, conditional expression (`IfExp`) and `if` clause in a comprehension, plus `len(values) - 1` for each `and`/`or` (`BoolOp`).

In [ ]:
import ast

def complexity(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_complexity():\n    \"paths through each function\"\n    c = complexity(open(\"planner.py\").read())\n    assert c == {\"straight_km\": 1, \"astar\": 7, \"describe\": 1, \"pick\": 9, \"_cache_key\": 1}\n    src = \"def f(x):\\n    try:\\n        y = [a for a in x if a and a > 1]\\n    except ValueError:\\n        y = 0 if x else 1\\n    return y\\n\"\n    assert complexity(src) == {\"f\": 5}")

### Exercise: Docstrings and type hints

Write `missing_docstrings(source)`: the names of top-level **public** functions and classes (not starting with `_`) that have no docstring, in order. Then `missing_hints(source)`: the names of public top-level functions where any parameter other than `self` or `cls` has no annotation, or there's no return annotation, in order.

In [ ]:
import ast

def missing_docstrings(source):
    ___

def missing_hints(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_interface():\n    \"what other modules rely on\"\n    src = open(\"planner.py\").read()\n    assert missing_docstrings(src) == [\"astar\", \"pick\"]\n    assert missing_hints(src) == [\"straight_km\", \"astar\", \"pick\"]\n    ok = 'class A:\\n    \"\"\"A thing.\"\"\"\\n\\ndef f(x: int) -> int:\\n    \"\"\"Double.\"\"\"\\n    return 2 * x\\n\\ndef _g(y):\\n    return y\\n'\n    assert missing_docstrings(ok) == [] and missing_hints(ok) == []")

### Exercise: Long functions

Write `function_lengths(source)`: a dict from each top-level function's name to its length in lines (from its `def` line to its last line, using `lineno` and `end_lineno`). Then `too_long(source, limit=30)`: the names longer than `limit`, longest first (ties by name).

In [ ]:
import ast

def function_lengths(source):
    ___

def too_long(source, limit=30):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_length():\n    \"lines per function\"\n    src = open(\"planner.py\").read()\n    assert function_lengths(src) == {\"straight_km\": 3, \"astar\": 19, \"describe\": 4, \"pick\": 12, \"_cache_key\": 2}\n    assert too_long(src, 10) == [\"astar\", \"pick\"] and too_long(src) == []")

### Exercise: Test smells

Write `test_smells(source)` for a test module: a list of `(test_name, smell)` for top-level functions whose names start with `test`, in order, with each test's smells in this order: `"no assertion"` (no `assert` statement and no call to `pytest.raises`); `"sleeps"` (a call to `time.sleep` or `sleep`); `"catches everything"` (a call to `pytest.raises` whose first argument is the name `Exception` or `BaseException`); and `"unseeded random"` (a call to `random.Random` with no arguments).

In [ ]:
import ast

def test_smells(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def check_smells():\n    \"found in this sprint's tests\"\n    assert test_smells(open(\"test_planner.py\").read()) == [\n        (\"test_astar_runs\", \"no assertion\"), (\"test_pick_waits_for_cache\", \"sleeps\"),\n        (\"test_pick_errors\", \"catches everything\"), (\"test_random_routes\", \"unseeded random\")]\n\ndef check_clean():\n    \"nothing to say\"\n    clean = \"import pytest\\nimport random\\n\\ndef test_a():\\n    assert 1 == 1\\n\\ndef test_b():\\n    with pytest.raises(ValueError):\\n        int('x')\\n\\ndef test_c():\\n    assert random.Random(4).random() < 1\\n\\ndef helper():\\n    pass\\n\"\n    assert test_smells(clean) == []\n\ndef test_all():\n    \"both\"\n    check_smells()\n    check_clean()")

### Bug hunt: A sort that reached the caller

`cheapest` sorts the list it's given **in place**, so the caller's list of routes changes order behind its back: it prints `['short', 'long']` for the caller's list, which should still be in its original order. Sort a copy (`sorted`), so it prints `['long', 'short']`.

In [ ]:
def cheapest(routes):
    routes.sort(key=lambda r: r["km"])
    return routes[0]

offered = [{"name": "long", "km": 9}, {"name": "short", "km": 4}]
cheapest(offered)
print([r["name"] for r in offered])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"don't change your arguments\"\n    expect_output(\"['long', 'short']\")")

### Bug hunt: Every factor is the last one

The cost functions are made in a loop, and each lambda looks up `factor` when it's **called**, by which time the loop has finished: every one uses 3.0 and it prints `[6.0, 6.0]`. Bind the current value (a default argument), so it prints `[2.0, 6.0]`.

In [ ]:
costs = []
for factor in (1.0, 3.0):
    costs.append(lambda km: km * factor)
print([cost(2.0) for cost in costs])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"late binding\"\n    expect_output(\"[2.0, 6.0]\")")

### Puzzle: pick, as a table

Arrange a simpler `pick`: a table of key functions instead of a chain of `elif`s (complexity 2 instead of 9). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the cyclomatic complexity of a function with one if, one for and one "a and b and c"
q = ___   # the test smell in: with pytest.raises(Exception): ...
r = ___   # True or False: a private helper like _cache_key needs a docstring by this standard
s = ___   # the minimum number of tests to run every branch of a function of complexity 4
t = ___   # what late binding makes a lambda in a loop use: "the value at creation" or "the value when called"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 5 and q == \"catches everything\" and r is False, \"p, q and r\"\n    assert s == 4 and t == \"the value when called\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a quality report

Write `quality_report(source, test_source, max_complexity=7, max_lines=15)` using the given helpers. Return `(findings, score)`: `findings` is a sorted list of strings: `"complexity N: NAME"` for functions above `max_complexity`; `"long (N lines): NAME"` for functions above `max_lines`; `"no docstring: NAME"`; `"no type hints: NAME"`; and `"test smell (SMELL): TEST"` for each test smell. `score` starts at 100 and loses 10 per complexity or length finding, 5 per docstring or hint finding, and 10 per test smell, never below 0.

In [ ]:
import ast

def complexity(source):
    result = {}
    for node in ast.parse(source).body:
        if isinstance(node, ast.FunctionDef):
            score = 1
            for inner in ast.walk(node):
                if isinstance(inner, (ast.If, ast.For, ast.While, ast.ExceptHandler, ast.IfExp)):
                    score += 1
                elif isinstance(inner, ast.BoolOp):
                    score += len(inner.values) - 1
                elif isinstance(inner, ast.comprehension):
                    score += len(inner.ifs)
            result[node.name] = score
    return result

def function_lengths(source):
    return {n.name: n.end_lineno - n.lineno + 1 for n in ast.parse(source).body if isinstance(n, ast.FunctionDef)}

def missing_docstrings(source):
    return [n.name for n in ast.parse(source).body
            if isinstance(n, (ast.FunctionDef, ast.ClassDef)) and not n.name.startswith("_") and ast.get_docstring(n) is None]

def missing_hints(source):
    out = []
    for n in ast.parse(source).body:
        if isinstance(n, ast.FunctionDef) and not n.name.startswith("_"):
            if any(a.annotation is None and a.arg not in ("self", "cls") for a in n.args.args) or n.returns is None:
                out.append(n.name)
    return out

def test_smells(source):
    found = []
    for node in ast.parse(source).body:
        if not (isinstance(node, ast.FunctionDef) and node.name.startswith("test")):
            continue
        text = ast.unparse(node)
        if "assert " not in text and "pytest.raises" not in text:
            found.append((node.name, "no assertion"))
        if "sleep(" in text:
            found.append((node.name, "sleeps"))
        if "pytest.raises(Exception)" in text:
            found.append((node.name, "catches everything"))
        if "random.Random()" in text:
            found.append((node.name, "unseeded random"))
    return found

def quality_report(source, test_source, max_complexity=7, max_lines=15):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_report():\n    \"this sprint's planner\"\n    findings, score = quality_report(open(\"planner.py\").read(), open(\"test_planner.py\").read())\n    assert findings == [\"complexity 9: pick\", \"long (19 lines): astar\", \"no docstring: astar\", \"no docstring: pick\",\n                        \"no type hints: astar\", \"no type hints: pick\", \"no type hints: straight_km\",\n                        \"test smell (catches everything): test_pick_errors\", \"test smell (no assertion): test_astar_runs\",\n                        \"test smell (sleeps): test_pick_waits_for_cache\", \"test smell (unseeded random): test_random_routes\"]\n    assert score == 15\n\ndef test_clean_and_floor():\n    \"a clean module, and a terrible one\"\n    clean = 'def f(x: int) -> int:\\n    \"\"\"Double.\"\"\"\\n    return 2 * x\\n'\n    assert quality_report(clean, \"def test_f():\\n    assert f(2) == 4\\n\") == ([], 100)\n    awful = \"\\n\".join(f\"def f{i}(x):\\n    if x:\\n        pass\" for i in range(30))\n    assert quality_report(awful, \"\")[1] == 0")

## Warm-up

The syntax tree tells you each function's size, how many decisions it makes, and whether it explains itself. Open `planner.py` and `test_planner.py` above: they're from this sprint's pull requests, and they're what you'll review today.

## Reviewing algorithm changes

For a change to an algorithm, ask, in addition to last week's list:

- **Is it still correct on the awkward inputs?** No route, start equal to end, ties, a disconnected map. The tests from yesterday (oracles, invariants, metamorphic relations, the regression suite) should already say; check they ran.
- **What does it cost?** Read the loops: a `list` membership test or `sorted()` inside a loop over every junction turns a fast search slow. Ask for the growth test.
- **What does it assume?** A* assumes paths are never shorter than the straight line; preferences assume factors of at least 1. Is the assumption written down, and checked?
- **Is the change contained?** The search should change in one module; the API and CLI shouldn't notice.

## Measuring complexity

**Cyclomatic complexity** counts the independent paths through a function: start at 1 and add 1 for each decision (`if`, `elif`, `for`, `while`, each `except`, each extra condition in an `and` or `or`, each `if` in a comprehension, and each conditional expression). It's also the minimum number of tests needed to run every branch. Functions above about 10 are hard to understand and test; `pick` in `planner.py` is a candidate. The fix is usually a table instead of a chain of `elif`s, or smaller functions.

**Length** matters too: a function over about 30 to 40 lines usually does several things. Both are easy to measure with `ast` (each node knows its `lineno` and `end_lineno`), which means they can be checked in CI and discussed only when they're exceeded.

## The public interface

Functions and classes without a leading underscore are the module's **public interface**: other modules (and your future self) rely on them. Each should have a **docstring** saying what it does (not how), and **type hints** (Week 33) on its parameters and return value, so readers and tools know what goes in and comes out. Private helpers (`_cache_key`) can be terser. In `planner.py`, `astar` and `pick` have neither.

## Reviewing the tests

Tests deserve review as much as code, because a bad test is worse than none: it gives confidence it hasn't earned. The usual **test smells**:

- **No assertion**: `test_astar_runs` only checks that nothing crashed.
- **Sleeping**: `time.sleep` makes tests slow and usually hides a timing dependency that should be injected.
- **Catching everything**: `pytest.raises(Exception)` passes for **any** error, including a typo; name the exception you expect.
- **Unseeded randomness**: `random.Random()` without a seed gives a different test each run.
- **Testing the mock**, and **assertions in loops** that might run zero times.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Measure** your capstone with today's tools (or ruff's complexity rule, `C901`, and a docstring rule set): note the three worst functions.
2. **Simplify one** (a table instead of a chain, or smaller functions), with its tests passing before and after.
3. **Review your tests** for smells and fix every one.
4. **Add docstrings and hints** to your public interface, and the checks to CI.
5. **Merge** and update the burndown.

## Before you go

Tomorrow is the **sprint review** for sprint 2: what was built, the numbers, and the plan for the last build sprint.